# 04 · RQ3b: faulty sensor inputs and the edge filter

Synthetic faults (spike, stuck, dropout; Ni et al. / Sharma et al. taxonomy) are injected into the TEST inputs; three conditions are compared: clean, faulty with the edge filter off, faulty with the edge filter on. Targets are never touched. This measures forecast robustness, not fault-detection quality. Run notebooks 02 and 03 first.

*Estimated time: QUICK: 1-2 min (1 seed); full: about 3 min per 5 seeds.*

In [ ]:
# --- parameters -------------------------------------------------------------------------
QUICK = True        # 2 stations, short run (< 5 min); set False for the full 12-station run
USE_DRIVE = False   # Colab only: mount Google Drive to keep results between sessions
RUN_OPTUNA = False  # hyper-parameter search (notebook 02 onwards)
SEEDS = [0]


In [ ]:
# --- setup (runs on Colab, skipped locally) ---------------------------------------------
import os
import subprocess
import sys
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/furkantopcuu/airq-iot.git"
if IN_COLAB:
    if not os.path.exists("/content/airq-iot"):
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/airq-iot"], check=True)
    os.chdir("/content/airq-iot")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
    sys.path.insert(0, "/content/airq-iot/src")  # no restart needed after the editable install


In [ ]:
from airq.config import load_config
from airq.env import mount_drive_if_wanted
from airq.pipelines.colab import env_summary

mount_drive_if_wanted(USE_DRIVE)
cfg = load_config()
env_summary(cfg)


In [ ]:
from airq.pipelines.edge import run as fit_edge
from airq.pipelines.rq3b import run_rq3b

fit_edge(cfg)  # edge thresholds fitted on train
summary = run_rq3b(cfg, quick=QUICK, seeds=SEEDS, force=QUICK)  # full run: guarded
summary["persistence_coverage"]


In [ ]:
import pandas as pd
from IPython.display import Image, display
from airq.env import resolve_paths
from airq.pipelines.figures import rq3b

tables = resolve_paths(cfg["paths"])["reports"] / "tables"
rob = pd.read_csv(tables / "rq3b_robustness.csv")
display(rob[rob["scope"].isin(["all", "affected"])].round(2))
display(pd.read_csv(tables / "rq3b_detection.csv").round(2))
for p in rq3b(cfg):
    if p.suffix == ".png":
        display(Image(filename=str(p)))


In [ ]:
# --- export: reports/ and models/ -> airq_artifacts_<date>.zip --------------------------
from airq.pipelines.colab import export_artifacts

zip_path = export_artifacts(cfg)
print("Wrote", zip_path.name)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
# locally:  python -m scripts.import_artifacts <zip>   (also runs the manifest check)
